In [151]:
import pandas as pd

In [152]:
df = pd.read_csv('/Users/DakshSoni/Downloads/IMDB Dataset.csv')

In [153]:
df.shape

(50000, 2)

In [154]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [155]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [156]:
df.drop_duplicates(inplace = True)

In [157]:
df.shape

(49582, 2)

### Pre Processing

### 1. Converting to Lowercase

In [158]:
df["review"] = df["review"].str.lower()  

### 2. Removing URL

In [159]:
import re

# sample_text = "daksh is good"  # daksh -> he
# new_text = re.sub("daksh", "he", sample_text)  # pattern, word inplace of replace, string
# new_text


In [160]:
def remove_urls(text):
    text = re.sub(r"http\S+", "", text) 
    return text

df["review"] = df["review"].apply(remove_urls)    

### 3. Remove punctuations

In [161]:
def remove_punctutations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text) # A-Z,a-z,0-9,\s   anything else is called a punctuation
    return text

df["review"] = df["review"].apply(remove_punctutations)   

### 4. Remove HTML 

In [162]:
def remove_HTML(text):
    text = re.sub(r"<.*?>", "", text) 
    return text

df["review"] = df["review"].apply(remove_HTML)   

### 5. Remove Stopwords

In [163]:
import nltk

nltk.download("punkt") # official tokenizer
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /Users/dakshsoni/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/dakshsoni/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/dakshsoni/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [164]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [165]:
# sample_text = "I like playing Cricket!!!!"
# tokens  = word_tokenize(sample_text)
# tokens 

In [166]:
def removes_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")
            
    return text        

df["review"] = df["review"].apply(removes_stopwords)

In [167]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6. Stemming

In [168]:
# running -> run
# played -> play
# PorterStemming

from nltk.stem import PorterStemmer

In [169]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)


    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)    

In [170]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7. Encoding

In [171]:
from sklearn.preprocessing import LabelEncoder

le  = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [172]:
y = df["sentiment"]

In [173]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8. Vectorization TF-IDF

In [174]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features = 5000)

X = tf.fit_transform(df["review"])

In [175]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057169 stored elements and shape (49582, 5000)>
  Coords	Values
  (0, 3538)	0.05515484681268887
  (0, 2869)	0.09361182809242374
  (0, 4940)	0.11467310366614668
  (0, 3003)	0.47200539890110405
  (0, 1275)	0.1357698919196183
  (0, 2289)	0.049500237517476293
  (0, 1933)	0.0791260308382
  (0, 3550)	0.0963974330192639
  (0, 1362)	0.06162489377343992
  (0, 1963)	0.061560444992697486
  (0, 218)	0.08588920995304898
  (0, 1620)	0.0738170550485134
  (0, 4368)	0.041994187696759305
  (0, 4170)	0.17799685402440263
  (0, 3692)	0.033532198172897175
  (0, 4737)	0.26798942924092045
  (0, 3804)	0.04427609784380831
  (0, 4769)	0.05877405881441711
  (0, 1739)	0.037520883911174724
  (0, 4496)	0.07614066339174266
  (0, 3856)	0.17537900435282314
  (0, 1630)	0.06142445471882175
  (0, 1862)	0.07433134577032253
  (0, 3329)	0.06406818508428483
  (0, 3332)	0.0844754682576354
  :	:
  (49581, 4890)	0.10682334916138103
  (49581, 1542)	0.17584072573791829

### 9. Dataset and DataLoaders

In [176]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 42)

In [177]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3248334 stored elements and shape (39665, 5000)>

In [178]:
X_test.shape

(9917, 5000)

In [179]:
import torch
import numpy as mp
from torch.utils.data import TensorDataset, DataLoader

In [180]:
X_train = X_train.toarray() # convert the sparsed matrix to numpy array
X_test = X_test.toarray() # convert the sparsed matrix to numpy array

In [181]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [182]:
train_Loader = DataLoader(train_set, shuffle = True, batch_size = 64)
test_Loader = DataLoader(test_set, shuffle = True, batch_size = 64)

### Build RNN

In [183]:
import torch.nn as nn
import torch.optim as optim

In [184]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size = 128, num_layers = 1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first = True)

        # fully connected Layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self,x):
        # optimal => shape (num of layers, batch_size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _  = self.rnn(x, h0)
        # 1st value = hidden state of all the timestemps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [185]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss() # Binary Cross entropy Loss is use when we have 2 outputs 
optimizer = optim.Adam(model.parameters())

### Training the RNN

In [186]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_Loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add a singleton direction cuz the the mnodel is expecting us to give a 3D input 

        outputs = model(Xb)

        outputs = torch.sigmoid(outputs.squeeze()) # to reduce the dimention and this output gives us probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update 

    print(f"{epoch + 1}/{epochs} and loss = {loss.item()}")    

1/10 and loss = 0.320037305355072
2/10 and loss = 0.19425158202648163
3/10 and loss = 0.1166582852602005
4/10 and loss = 0.3102969229221344
5/10 and loss = 0.20784662663936615
6/10 and loss = 0.2414046674966812
7/10 and loss = 0.09512662887573242
8/10 and loss = 0.23423628509044647
9/10 and loss = 0.29173752665519714
10/10 and loss = 0.3126175105571747


In [188]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    total_vals = 0

    for Xb, yb in test_Loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        total_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/total_vals *100}")    

accuracy = 85.6609861853383
